In [1]:
pip install ydata-profiling[pyspark]

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession
from ydata_profiling import ProfileReport

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark profiling example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [15]:
# Copy File to bronze layer
from os import PathLike
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

from_path = "/home/jovyan/projeto/csvs/amazon_books.csv"
to_path = "/projeto/bronze/amazon_books.csv"

client.delete(to_path)
client.upload(to_path, from_path)

In [16]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/amazon_books.csv"

In [17]:
amazon_books_df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [ ]:
# Note that some profiling operations can resulte in errors due to bad loading options. 
# It is a good praticce start by inspect the schema and a data sample. 
amazon_books_df.printSchema()
amazon_books_df.show()

In [ ]:
amazon_titles_df.select("seasons").distinct().count()

In [ ]:
amazon_books_df.filter(amazon_books_df["discount"].isNull()).count()

In [ ]:
from pyspark.sql import functions as F

# número total de linhas
total_rows = amazon_titles_df.count()

# para cada coluna, contar valores repetidos
for col in amazon_titles_df.columns:
    distinct_count = amazon_titles_df.select(col).distinct().count()
    repeated_count = total_rows - distinct_count
    print(f"{col}: {repeated_count} valores repetidos")

In [8]:
from pyspark.sql import functions as F

# agrupa pelos títulos e conta quantas vezes aparecem
duplicados = (
    amazon_titles_df.groupBy("title")
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
)

# mostra um exemplo de filme repetido
duplicados.show(1)

NameError: name 'amazon_titles_df' is not defined

In [11]:
from pyspark.sql import functions as F
min_id = amazon_titles_df.agg(F.min("tmdb_score").alias("min_id")).collect()[0]["min_id"]
amazon_titles_df.select(F.min("tmdb_score"), F.max("tmdb_score")).show()

+---------------+---------------+
|min(tmdb_score)|max(tmdb_score)|
+---------------+---------------+
|            0.8|           10.0|
+---------------+---------------+



In [12]:
#In case of error select a subset of columns until you find the column that causes that.
#For start we can use describe as starting point for data profiling
#For this example the column summary was removed due to a conflit with the first describe column "summary"
amazon_books_df.describe(['asin','ISBN10','title','brand','availability','discount','initial_price','final_price','currency','rating','reviews_count','categories','best_sellers_rank','timestamp','url']).toPandas()

,summary,asin,ISBN10,title,brand,availability,discount,initial_price,final_price,currency,rating,reviews_count,categories,best_sellers_rank,url
0,count,2269,1429,2269,2268,1375,1177,1177,1392,2269,2269,2269,2269,2268,2269
1,mean,1.0161135680046693E9,1.2057561119344897E12,1984.0,None,None,8.202361937128307,21.755284621920445,13.380028735632195,None,None,21497.73821066549,None,None,None
2,stddev,8.33623777562542E8,3.21538542687871E12,0.0,None,None,10.211126760677864,15.488614194564692,8.116235656737315,None,None,16108.019322281432,None,None,None
3,min,0007350813,0007350813,10-Day Green Smoothie Cleanse,"A. G. Riddle (Author), Stephen Bel Davies (Nar...",Available to ship in 1-2 days.,0.5,3.49,1.99,USD,3.4 out of 5 stars,10010,"[""Books"",""Arts & Photography"",""Drawing""]","[{""category"":""Audible Books & Originals / Arts...",https://www.amazon.com/dp/0007350813
4,max,B09N1L9QK6,�� 1641526270,milk and honey,by Yuval Harari (Author),Usually ships within 8 days.,282.49,299.0,132.99,USD,4.9 out of 5 stars,196572,"[""Toys & Games"",""Learning & Education"",""Electr...","[{""category"":""Toys & Games"",""rank"":72},{""categ...",https://www.amazon.com/dp/B09N1L9QK6


In [13]:
#Select the columns to profile. 
df_to_profile = amazon_books_df.select("asin","ISBN10","title","brand","availability","discount","initial_price","final_price","currency","rating","reviews_count","categories","best_sellers_rank","url")

In [21]:
#create the Profile report using the ydata profiling tool. More info at https://docs.profiling.ydata.ai/
report = ProfileReport(df_to_profile)

In [22]:
#save profiling report in a file
report.to_file('amazon_books.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
#close spark session
spark.stop()